In [1]:
# Cell 1: Imports and Setup
import sys
import os
import random
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from collections import defaultdict
from typing import List
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../src"))

from abstractions3d.primitives.shapes import Box3D
from abstractions3d.primitives.visualization import visualize_boxes_3d
from abstractions3d.dsl.core import Shape3D
from abstractions3d.dsl.nodes import Rect3D, Move3D, Union3D, SymTrans3D, SymRef3D
from abstractions3d.dsl.instantiation import instantiate_3d
from abstractions3d.data.blueprint import Table3D, Chair3D, Bench3D

# Set random seeds
random.seed(42)
torch.manual_seed(42)

In [2]:
# Cell 2: Helper Functions

# Sampling function
def sample_uniform(low: float, high: float) -> float:
    return random.uniform(low, high)

# Generate dataset
def generate_dataset(num_samples: int = 50) -> List[Shape3D]:
    dataset = []
    for _ in range(num_samples):
        table = Table3D(
            top_length=sample_uniform(3.0, 6.0),
            top_depth=sample_uniform(1.5, 3.0),
            top_thickness=sample_uniform(0.1, 0.3),
            leg_length=sample_uniform(0.2, 0.4),
            leg_depth=sample_uniform(0.2, 0.4),
            leg_height=sample_uniform(1.0, 2.0)
        )
        dataset.append(table)

        chair = Chair3D(
            seat_length=sample_uniform(1.0, 2.0),
            seat_depth=sample_uniform(1.0, 2.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=sample_uniform(1.0, 2.0),
            backrest_thickness=sample_uniform(0.1, 0.3)
        )
        dataset.append(chair)

        bench = Bench3D(
            seat_length=sample_uniform(2.0, 5.0),
            seat_depth=sample_uniform(0.5, 1.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=sample_uniform(0.5, 1.5),
            backrest_thickness=sample_uniform(0.1, 0.3)
        )
        dataset.append(bench)
    return dataset

In [3]:
# Cell 3: Utilities for Abstraction Discovery

# Singletons
def get_singletons(shapes):
    singletons = defaultdict(list)
    if isinstance(shapes, list):
        for s in shapes:
            res = get_singletons(s)
            for k, v in res.items():
                singletons[k] += v
        return singletons

    cls, params = shapes.param_tuple()
    singletons[cls.__name__].append(params)

    if hasattr(shapes, 'children'):
        for child in getattr(shapes, 'children', []):
            res = get_singletons(child)
            for k, v in res.items():
                singletons[k] += v
    return singletons

# Pairs
def get_pairs(shapes):
    pairs = defaultdict(list)
    if isinstance(shapes, list):
        for s in shapes:
            res = get_pairs(s)
            for k, v in res.items():
                pairs[k] += v
        return pairs

    if hasattr(shapes, 'children') and len(shapes.children) == 2:
        cls, (child1, child2) = shapes.param_tuple()
        type_str = f"{cls.__name__}({child1.param_tuple()[0].__name__},{child2.param_tuple()[0].__name__})"
        pairs[type_str].append(child1.param_tuple()[1] + child2.param_tuple()[1])
        for child in shapes.children:
            res = get_pairs(child)
            for k, v in res.items():
                pairs[k] += v
    return pairs

# Prepare Autoencoder data
def prepare_autoencoder_train_data(parameters, mask):
    tensor = torch.tensor(parameters, dtype=torch.float32)
    masked_data = tensor[mask]
    if len(masked_data) == 0:
        return None
    dataset = TensorDataset(masked_data)
    return DataLoader(dataset, batch_size=64, shuffle=True)

# Check well-explained
def is_well_explained(parameters, model, threshold=0.01):
    with torch.no_grad():
        recon = model(parameters)
        error = torch.max(torch.abs(recon - parameters), dim=1).values
        return error < threshold

In [4]:
# Cell 4: Abstraction Node

class Abstraction(Shape3D):
    def __init__(self, type_list, float_parameters, other_parameters, model):
        self.type_list = type_list
        self.float_parameters = float_parameters
        self.other_parameters = other_parameters
        self.model = model
        self.children = [p for p in other_parameters if isinstance(p, Shape3D)]

    def __str__(self):
        s = "Abstraction(\n"
        for c in self.children:
            s += "  " + str(c) + "\n"
        for f in self.float_parameters:
            s += f"  {f}\n"
        for o in self.other_parameters:
            if not isinstance(o, Shape3D):
                s += f"  {o}\n"
        s += ")"
        return s

    def expand(self):
        self.model.eval()
        float_tensor = torch.tensor(self.float_parameters, dtype=torch.float32).unsqueeze(0)
        decoder_output = self.model(float_tensor)
        expanded_float_parameters = decoder_output.squeeze(0).tolist()

        full_parameter_list = []
        i_floats = 0
        i_others = 0
        for t in self.type_list:
            if t == float:
                full_parameter_list.append(expanded_float_parameters[i_floats])
                i_floats += 1
            else:
                full_parameter_list.append(self.other_parameters[i_others])
                i_others += 1

        return instantiate_3d(self.type_list, full_parameter_list)

In [5]:
# Cell 5: Train Autoencoders

def train_abstractions(structures, retrain_iterations=2, error_threshold=0.01):
    losses = defaultdict(list)
    models = {}

    for structure_name, parameters in structures.items():
        float_indices = [i for i, t in enumerate(parameters[0]) if isinstance(t, float)]
        if not float_indices:
            continue

        float_params = [[p[i] for i in float_indices] for p in parameters]
        float_tensor = torch.tensor(float_params, dtype=torch.float32)
        well_explained = torch.ones(len(float_tensor), dtype=torch.bool)

        for _ in range(retrain_iterations):
            train_loader = prepare_autoencoder_train_data(float_tensor, well_explained)
            if train_loader is None:
                print(f"Skipping {structure_name}: no well-explained samples")
                break

            model = nn.Sequential(
                nn.Linear(len(float_indices), 32),
                nn.ReLU(),
                nn.Linear(32, 32),
                nn.ReLU(),
                nn.Linear(32, len(float_indices))
            )
            optimizer = AdamW(model.parameters(), lr=0.001, weight_decay=0.05)
            loss_fn = lambda recon, x: torch.mean(torch.max(torch.abs(recon - x), dim=1).values)

            model.train()
            for epoch in range(100):
                running_loss = 0.0
                for batch in train_loader:
                    x = batch[0]
                    optimizer.zero_grad()
                    recon = model(x)
                    loss = loss_fn(recon, x)
                    loss.backward()
                    optimizer.step()
                    running_loss += loss.item()
                losses[structure_name].append(running_loss / len(train_loader))

            model.eval()
            well_explained = is_well_explained(float_tensor, model, threshold=error_threshold)

            # Ensure at least one True to avoid empty DataLoader
            if torch.sum(well_explained) == 0:
                well_explained[0] = True

        models[structure_name] = model
    return models, losses

In [6]:
def integrate_abstractions(shape_or_list, models, error_threshold=0.01):
    """Recursively replace well-explained subtrees with Abstraction nodes."""
    if isinstance(shape_or_list, list):
        return [integrate_abstractions(s, models, error_threshold) for s in shape_or_list]

    # If shape is already None or not a Shape3D, return as is
    if shape_or_list is None or not isinstance(shape_or_list, Shape3D):
        return shape_or_list

    shape = shape_or_list

    # Safely get param_tuple
    try:
        cls, parameters = shape.param_tuple()
    except Exception:
        return shape  # return original shape if param_tuple fails

    type_str = cls.__name__

    # Extract float parameters
    float_params = [p for p in parameters if isinstance(p, float)]
    other_params = [p for p in parameters if not isinstance(p, float)]

    fits_well = False
    if float_params and type_str in models:
        float_tensor = torch.tensor([float_params], dtype=torch.float32)
        with torch.no_grad():
            recon = models[type_str](float_tensor)
            error = torch.max(torch.abs(recon - float_tensor))
            fits_well = error < error_threshold

    if fits_well:
        type_list = [cls] + [type(p) for p in parameters]
        new_shape = Abstraction(type_list, float_params, other_params, models[type_str])
    else:
        new_shape = cls(*parameters)

    # Recursively integrate into children if available
    if hasattr(new_shape, 'children') and isinstance(new_shape.children, list):
        new_children = [integrate_abstractions(c, models, error_threshold) for c in new_shape.children]
        new_shape.children = new_children

    return new_shape

In [7]:
# Cell 6: Full Abstraction Pipeline

def abstraction_pipeline(dataset, error_threshold=0.01):
    singletons = get_singletons(dataset)
    pairs = get_pairs(dataset)

    singleton_models, _ = train_abstractions(singletons, error_threshold=error_threshold)
    pair_models, _ = train_abstractions(pairs, error_threshold=error_threshold)

    dataset_singleton_abstracted = integrate_abstractions(dataset, singleton_models, error_threshold)
    dataset_fully_abstracted = integrate_abstractions(dataset_singleton_abstracted, pair_models, error_threshold)

    return dataset_fully_abstracted

In [8]:
# Cell 7: Run Pipeline and Visualize

# Generate dataset
dataset = generate_dataset(50)
print(f"Generated {len(dataset)} shapes.")

# Apply abstraction pipeline
abstracted_dataset = abstraction_pipeline(dataset)
print("Abstraction pipeline completed.")

# Visualize first 5 shapes
# for shape in abstracted_dataset[:5]:
#     boxes = shape.get_box3d_list()
#     visualize_boxes_3d(boxes)

Generated 150 shapes.


/var/folders/zb/9bmh3s9s3q5_mlv24by4t5x00000gn/T/ipykernel_68545/4289147842.py:45: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tensor = torch.tensor(parameters, dtype=torch.float32)
/var/folders/zb/9bmh3s9s3q5_mlv24by4t5x00000gn/T/ipykernel_68545/4289147842.py:45: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tensor = torch.tensor(parameters, dtype=torch.float32)


Abstraction pipeline completed.


In [11]:
abstracted_dataset[1].get_box3d_list()

In [12]:
# Function to collect Abstraction nodes recursively
def collect_abstractions(shapes):
    abstractions = []
    if isinstance(shapes, list):
        for s in shapes:
            abstractions.extend(collect_abstractions(s))
        return abstractions

    if isinstance(shapes, Abstraction):
        abstractions.append(shapes)

    if hasattr(shapes, 'children'):
        for child in shapes.children:
            abstractions.extend(collect_abstractions(child))

    return abstractions

In [13]:
abstracted_nodes = collect_abstractions(abstracted_dataset)
print(f"Total Abstraction nodes in dataset: {len(abstracted_nodes)}")

# Optionally, inspect the first few
for i, node in enumerate(abstracted_nodes[:5]):
    print(f"\nAbstraction node {i+1}:")
    print(node)

Total Abstraction nodes in dataset: 313

Abstraction node 1:
Abstraction(
  Rect3D(
    4.918,
    0.155,
    1.538
)
  0.0
  1.7542024192598231
  0.0
)

Abstraction node 2:
Abstraction(
  Rect3D(
    1.892,
    0.327,
    1.087
)
  0.0
  0.9159659170044717
  0.0
)

Abstraction node 3:
Abstraction(
  Rect3D(
    1.892,
    1.027,
    0.140
)
  0.0
  1.592522174799194
  -0.4735856512460432
)

Abstraction node 4:
Abstraction(
  0.21785313677518175
  0.5032493798390305
  0.2618860913355653
)

Abstraction node 5:
Abstraction(
  Rect3D(
    3.950,
    1.306,
    0.240
)
  0.0
  1.4222911923676436
  -0.2664214306519815
)


In [14]:
# Function to check if a shape contains any Abstraction nodes
def has_abstraction(shape):
    if isinstance(shape, Abstraction):
        return True
    if hasattr(shape, 'children'):
        return any(has_abstraction(c) for c in shape.children)
    return False

# Collect indices
indices_with_abstraction = [i for i, s in enumerate(abstracted_dataset) if has_abstraction(s)]
print("Shape indices containing Abstraction nodes:", indices_with_abstraction)
print("Total shapes with Abstraction nodes:", len(indices_with_abstraction))

Shape indices containing Abstraction nodes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]
Total shapes with Abstraction nodes: 149


In [16]:
print(dataset[0])

Union3D(
    Move3D(
        Rect3D(
            4.918,
            0.155,
            1.538
        ),
        0.000,
        1.754,
        0.000
    ),
    SymRef3D(
        SymRef3D(
            Move3D(
                Rect3D(
                    0.245,
                    1.677,
                    0.347
                ),
                2.287,
                0.838,
                0.545
            ),
            x
        ),
        z
    )
)


In [17]:
print(abstracted_dataset[0])

Union3D(
    Abstraction(
      Rect3D(
        4.918,
        0.155,
        1.538
    )
      0.0
      1.7542024192598231
      0.0
    ),
    SymRef3D(
        SymRef3D(
            Move3D(
                Rect3D(
                    0.245,
                    1.677,
                    0.347
                ),
                2.287,
                0.838,
                0.545
            ),
            x
        ),
        z
    )
)


In [18]:
abstracted_dataset[0].get_box3d_list()

In [19]:
def count_nodes(shape):
    if isinstance(shape, list):
        return sum(count_nodes(s) for s in shape)
    cnt = 1
    if hasattr(shape, 'children'):
        cnt += sum(count_nodes(c) for c in shape.children)
    return cnt

# Count nodes before and after abstraction
total_before = count_nodes(dataset)
total_after = count_nodes(abstracted_dataset)

print("Total nodes before abstraction:", total_before)
print("Total nodes after abstraction:", total_after)

Total nodes before abstraction: 1350
Total nodes after abstraction: 1350
